## Manual analysis of total_sources 

List of sources that were selected to be part of our sample from a manual whitelist feature analysis and expansion to the whole raw sample 

### Set-up


In [1]:
import gc
import glob
import json
import os
from pathlib import Path

# Bridage des threads pour Numpy, Pandas et Scikit-Learn (laisse le CPU à DuckDB)
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"

import duckdb
from IPython.display import HTML, display
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns

# Configuration esthétique globale
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.family"] = "sans-serif"
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 30)

print("✔ Librairies importées et environnement configuré.")

✔ Librairies importées et environnement configuré.


In [2]:
# ==============================================================================
# 1. DÉFINITION DES CHEMINS (À adapter si tes fichiers sont dans un autre dossier)
# ==============================================================================
DATA_DIR = Path("/data/gdelt/gdelt_parquet_db")
SOURCE_MAP_PATH = Path("/data/gdelt/gdelt_sources_mapping.json")
RETAINED_IDS_PATH = Path("liste_ids_retenus.txt")
DOMAINS_PARQUET_PATH = "data/domains/domains_*.parquet"

# ==============================================================================
# 2. INITIALISATION ET PARAMÉTRAGE DE DUCKDB
# ==============================================================================
con = duckdb.connect()

# Réglage musclé pour ton grand serveur partagé (128 Go RAM / 16 threads en standard)
con.execute("PRAGMA memory_limit='128GB'")
con.execute("PRAGMA threads=16")

# 3. Chargement du dictionnaire JSON (Traduction ID <-> Nom de domaine)
with open(SOURCE_MAP_PATH, "r", encoding="utf-8") as f:
  source_map = json.load(f)

src_df = pd.DataFrame({
    "SourceCommonName_ID": [int(k) for k in source_map["id_to_source"].keys()],
    "SourceCommonName": list(source_map["id_to_source"].values()),
})
con.register("src_map", src_df)

# 4. Chargement de TA LISTE PROPRE d'ID retenus dans une table DuckDB dédiée
con.execute(f"""
    CREATE OR REPLACE TABLE retained_ids AS 
    SELECT column0::BIGINT AS id 
    FROM read_csv('{RETAINED_IDS_PATH}', header=False)
""")

nb_ids = con.execute("SELECT COUNT(*) FROM retained_ids").fetchone()[0]
print(
    f"✔ DuckDB initialisé. Table 'retained_ids' chargée ({nb_ids:,} médias"
    " légitimes)."
)

✔ DuckDB initialisé. Table 'retained_ids' chargée (13,334 médias légitimes).


In [3]:
glob_pattern = str(DATA_DIR / "gdelt_*.parquet")

print(
    "⏳ Création de la vue maîtresse 'gkg_clean' (Filtrage sur échantillon"
    " propre & enrichissement)..."
)

con.execute(f"""
    CREATE OR REPLACE VIEW gkg_clean AS
    
    WITH raw_filtered AS (
        -- 1. Nettoyage initial : dates valides, exclusion ligne corrompue et thèmes vides
        SELECT *
        FROM read_parquet('{glob_pattern}')
        WHERE regexp_matches(CAST(DATE AS VARCHAR), '^\\d{{14}}$')
          AND GKGRECORDID != '20210925181500-T1111'
          AND EnhancedThemes IS NOT NULL 
          AND EnhancedThemes != ''
    ),
    
    repaired AS (
        -- 2. Réparation de l'identifiant (si 0 ou NULL, on tente de matcher le nom de domaine via src_map)
        SELECT 
            r.* EXCLUDE (SourceCommonName_ID),
            CASE 
                WHEN COALESCE(r.SourceCommonName_ID, 0) = 0 THEN m.SourceCommonName_ID
                ELSE r.SourceCommonName_ID
            END AS SourceCommonName_ID
        FROM raw_filtered r
        LEFT JOIN src_map m 
          ON RTRIM(regexp_extract(r.DocumentIdentifier, 'https?://(?:www\\.)?([^/?:]+)', 1), '.') = m.SourceCommonName
    )
    
    -- 3. FILTRAGE STRICT SUR L'ÉCHANTILLON ET ENRICHISSEMENT WIKIDATA
    SELECT 
        rep.*,
        w.medialabel,
        w.typelabel,
        w.countrylabel,
        w.inception,
        CASE WHEN w.Src_ID IS NOT NULL THEN 1 ELSE 0 END AS is_wiki
    FROM repaired rep
    -- 🔥 LE FILTRE CLÉ : Jointure interne avec ta liste des sources retenues !
    INNER JOIN retained_ids rid 
      ON rep.SourceCommonName_ID = rid.id
    -- Enrichissement optionnel : si la source est dans Wikidata, on récupère ses labels
    LEFT JOIN (
        SELECT 
            id AS Src_ID, 
            medialabel, 
            typelabel, 
            countrylabel, 
            inception
        FROM read_parquet('{DOMAINS_PARQUET_PATH}')
        QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY inception ASC, countrylabel ASC) = 1
    ) w ON rep.SourceCommonName_ID = w.Src_ID;
""")

print(
    "✔ Vue 'gkg_clean' prête ! Elle ne contient que les articles de ton"
    " échantillon sélectionné."
)

⏳ Création de la vue maîtresse 'gkg_clean' (Filtrage sur échantillon propre & enrichissement)...
✔ Vue 'gkg_clean' prête ! Elle ne contient que les articles de ton échantillon sélectionné.


In [4]:
print("⏳ Calcul de la volumétrie globale sur ton échantillon...")

# 1. Requête rapide de vérification
df_check = con.execute("""
    SELECT 
        COUNT(*) AS total_articles_disponibles,
        COUNT(DISTINCT SourceCommonName_ID) AS medias_actifs,
        MIN(strptime(substr(CAST(DATE AS VARCHAR), 1, 8), '%Y%m%d')::DATE) AS date_min,
        MAX(strptime(substr(CAST(DATE AS VARCHAR), 1, 8), '%Y%m%d')::DATE) AS date_max,
        SUM(is_wiki) AS articles_avec_metadata_wiki
    FROM gkg_clean;
""").df()

# 2. Affichage du résumé
display(df_check.style.format({
    "total_articles_disponibles": "{:,.0f}",
    "medias_actifs": "{:,.0f}",
    "articles_avec_metadata_wiki": "{:,.0f}"
}))

# 3. Aperçu des 3 premiers articles au hasard pour vérifier la structure
print("\n👀 Aperçu de 3 articles au hasard dans la vue propre :")
display(con.execute("""
    SELECT 
        GKGRECORDID, 
        SourceCommonName_ID, 
        medialabel, 
        countrylabel, 
        substr(DocumentIdentifier, 1, 60) || '...' AS URL,
        substr(EnhancedThemes, 1, 80) || '...' AS Themes_extrait
    FROM gkg_clean 
    USING SAMPLE 3;
""").df())

⏳ Calcul de la volumétrie globale sur ton échantillon...


,total_articles_disponibles,medias_actifs,date_min,date_max,articles_avec_metadata_wiki
0,"1,136,023,125","13,334",2015-02-18 00:00:00,2026-06-19 00:00:00,"459,677,212"



👀 Aperçu de 3 articles au hasard dans la vue propre :


,GKGRECORDID,SourceCommonName_ID,medialabel,countrylabel,URL,Themes_extrait
0,20150222133000-T1584,28759,None,None,http://www.bresciaoggi.it/stories/2632_editori...,"GENERAL_GOVERNMENT,2312;TAX_FNCACT_EXECUTIVE,1..."
1,20150222133000-T2108,8745,None,None,http://www.internetajans.com/dis-haberler/dani...,"TAX_WORLDMAMMALS_HORSE,576;TAX_FNCACT_WOMAN,59..."
2,20150222134500-T2278,25709,None,None,http://www.dimokratiki.gr/22-02-2015/glezos-zi...,"POVERTY,1807;AUSTERITY,915;AUSTERITY,1084;AUST..."


In [5]:
def generer_statistiques(con, table_name="gkg_inter", col_source="domain", col_date="date"):
    """
    Génère un tableau de bord de statistiques descriptives réutilisable pour n'importe quelle vue GDELT.
    """
    
    # Extraction de l'année robuste : fonctionne si 'date' est un entier (20210512) ou un vrai TIMESTAMP
    year_expr = f"CAST(SUBSTRING(CAST({col_date} AS VARCHAR), 1, 4) AS INTEGER)"
    
    query = f"""
    -- 1. VOLUMÉTRIE GLOBALE
    SELECT '1. Volumétrie' AS Categorie, 'Nombre total d''articles' AS Indicateur, 
           CAST(COUNT(*) AS VARCHAR) AS Valeur, 'ℹ️' AS Statut 
    FROM {table_name}
    
    UNION ALL
    SELECT '1. Volumétrie', 'Nombre de sources uniques', 
           CAST(COUNT(DISTINCT {col_source}) AS VARCHAR), 'ℹ️' 
    FROM {table_name}
    
    UNION ALL
    SELECT '1. Volumétrie', 'Nombre moyen d''articles par source', 
           CAST(ROUND(COUNT(*) / CAST(COUNT(DISTINCT {col_source}) AS FLOAT), 1) AS VARCHAR), 'ℹ️' 
    FROM {table_name}

    -- 2. DYNAMIQUE TEMPORELLE
    UNION ALL
    SELECT '2. Dynamique Temporelle', 'Nombre moyen de sources actives par an', 
           CAST(ROUND(AVG(nb_src), 1) AS VARCHAR), 'ℹ️' 
    FROM (
        SELECT {year_expr} AS yr, COUNT(DISTINCT {col_source}) AS nb_src 
        FROM {table_name} 
        GROUP BY {year_expr}
    ) AS sub_sources_year

    UNION ALL
    SELECT '2. Dynamique Temporelle', 'Nombre moyen d''articles par an et par source', 
           CAST(ROUND(AVG(nb_arts), 1) AS VARCHAR), 'ℹ️' 
    FROM (
        SELECT {col_source}, {year_expr}, COUNT(*) AS nb_arts 
        FROM {table_name} 
        GROUP BY {col_source}, {year_expr}
    ) AS sub_arts_year

    -- 3. PÉRENNITÉ DES SOURCES
    UNION ALL
    SELECT '3. Pérennité', 'Répartition des sources selon le nb d''années d''activité', 
           STRING_AGG(nb_yrs || ' an(s): ' || nb_src || ' sources', '  |  ' ORDER BY nb_yrs), 'ℹ️' 
    FROM (
        SELECT nb_yrs, COUNT(*) AS nb_src 
        FROM (
            SELECT {col_source}, COUNT(DISTINCT {year_expr}) AS nb_yrs 
            FROM {table_name} 
            GROUP BY {col_source}
        ) AS sub_count_years
        GROUP BY nb_yrs
    ) AS sub_dist_years
    
    ORDER BY Categorie, Indicateur;
    """
    
    # Exécution de la requête
    df_stats = con.execute(query).df()
    
    # Application du style comme dans votre code
    styled_dashboard = df_stats.style.set_properties(**{'text-align': 'left'}, subset=['Categorie', 'Indicateur', 'Valeur'])\
                                     .set_properties(**{'text-align': 'center'}, subset=['Statut'])\
                                     .set_caption(f"<b>Statistiques descriptives - Table/Vue : {table_name}</b>")\
                                     .hide(axis="index")
    return styled_dashboard

### Random sample analysis 

#### Analysis of 100 sources, randomly chosen among all the selected sources 

In [6]:
print("Tirage aléatoire de 1000 sources dans l'échantillon retenu...\n")

# 1. Requête SQL : Jointure avec le dictionnaire des noms et tirage aléatoire
df_sample_100 = con.execute("""
    SELECT 
        r.id AS SourceCommonName_ID,
        COALESCE(m.SourceCommonName, 'Domaine inconnu (' || r.id || ')') AS SourceCommonName
    FROM retained_ids r
    LEFT JOIN src_map m ON r.id = m.SourceCommonName_ID
    ORDER BY random()
    LIMIT 100;
""").df()

# 2. Réinitialisation de l'index pour avoir un comptage propre de 1 à 100
df_sample_100.index = range(1, len(df_sample_100) + 1)
df_sample_100.index.name = '#'

# 3. AFFICHAGE EN GRILLE DE TEXTE (4 colonnes) pour un balayage visuel ultra-rapide
domaines = df_sample_100['SourceCommonName'].tolist()
n_cols = 4
n_rows = (len(domaines) + n_cols - 1) // n_cols

print("BALAYAGE RAPIDE — 1000 médias tirés au hasard :\n" + "="*85)
for r in range(n_rows):
    row_items = []
    for c in range(n_cols):
        idx = r + c * n_rows
        if idx < len(domaines):
            # On formate chaque nom sur 20 caractères pour aligner les colonnes
            row_items.append(f"{idx+1:3d}. {domaines[idx]:<18}")
    print(" | ".join(row_items))

print("\n" + "="*85)

# 4. AFFICHAGE DU DATAFRAME COMPLET (Pour inspection détaillée si besoin)
# On force Pandas à afficher les 100 lignes sans tronquer
with pd.option_context('display.max_rows', 1000):
    display(df_sample_100.style.set_properties(**{
        'font-weight': 'bold', 
        'text-align': 'left'
}))

Tirage aléatoire de 1000 sources dans l'échantillon retenu...

BALAYAGE RAPIDE — 1000 médias tirés au hasard :
  1. hoerzu.de          |  26. udinetoday.it      |  51. oilngold.com       |  76. workers.org       
  2. ilfordrecorder.co.uk |  27. lexo.al            |  52. fm4.orf.at         |  77. perishablenews.com
  3. kinja.com          |  28. brooklynrail.org   |  53. diariodecanoas.com.br |  78. parramattasun.com.au
  4. nnn.de             |  29. si24.it            |  54. varmatin.com       |  79. politonline.ru    
  5. knzr.com           |  30. 12newsnow.com      |  55. footwearnews.com   |  80. shropshirestar.com
  6. atlasinfo.fr       |  31. mirfieldreporter.co.uk |  56. primabrescia.it    |  81. obiectivtulcea.ro 
  7. ina-online.net     |  32. twincities.com     |  57. retfordtoday.co.uk |  82. designboom.com    
  8. blick.ch           |  33. annasronline.com   |  58. wyborcza.pl        |  83. 97xonline.com     
  9. diariodeburgos.es  |  34. kapanlagi.com      |  59. wcbe.

,SourceCommonName_ID,SourceCommonName
#,,
1,76591,hoerzu.de
2,7494,ilfordrecorder.co.uk
3,5110,kinja.com
4,25559,nnn.de
5,161148,knzr.com
6,30432,atlasinfo.fr
7,272203,ina-online.net
8,20126,blick.ch
9,108201,diariodeburgos.es


### Descriptive stats 

In [7]:
audit_query = """
-- 1. VOLUME ET UNICITÉ
SELECT '1. Volume & Unicité' AS Categorie, 'Nombre total d''articles' AS Indicateur, CAST(COUNT(*) AS VARCHAR) AS Valeur, 'ℹ️' AS Statut FROM gkg_clean
UNION ALL
SELECT '1. Volume & Unicité', 'Doublons sur GKGRECORDID', CAST(COUNT(*) - COUNT(DISTINCT GKGRECORDID) AS VARCHAR), CASE WHEN COUNT(*) = COUNT(DISTINCT GKGRECORDID) THEN '✅' ELSE '⚠️' END FROM gkg_clean

-- 2. CONFORMITÉ DES FORMATS
UNION ALL
SELECT '2. Conformité', 'URLs invalides (pas HTTP) sur Source=1', CAST(COUNT(*) AS VARCHAR), CASE WHEN COUNT(*) = 0 THEN '✅' ELSE '⚠️' END FROM gkg_clean WHERE SourceCollectionIdentifier = 1 AND DocumentIdentifier NOT ILIKE 'http%'
UNION ALL
SELECT '2. Conformité', 'IsTranslingual (valeurs hors 0/1)', CAST(COUNT(*) AS VARCHAR), CASE WHEN COUNT(*) = 0 THEN '✅' ELSE '⚠️' END FROM gkg_clean WHERE IsTranslingual NOT IN (0, 1)

-- 3. COMPLÉTUDE
UNION ALL
SELECT '3. Complétude (Critique)', 'GKGRECORDID / DATE / URL (% Vides)', CAST(ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM gkg_clean), 4) AS VARCHAR) || ' %', CASE WHEN COUNT(*) = 0 THEN '✅' ELSE '⚠️' END FROM gkg_clean WHERE GKGRECORDID IS NULL OR DATE IS NULL OR DocumentIdentifier IS NULL OR DocumentIdentifier = ''
UNION ALL
SELECT '3. Complétude (Information)', 'EnhancedThemes (% Vides)', CAST(ROUND(100.0 * SUM(CASE WHEN EnhancedThemes IS NULL OR EnhancedThemes = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS VARCHAR) || ' %', 'ℹ️' FROM gkg_clean
UNION ALL
SELECT '3. Complétude (Information)', 'EnhancedLocations (% Vides)', CAST(ROUND(100.0 * SUM(CASE WHEN EnhancedLocations IS NULL OR EnhancedLocations = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS VARCHAR) || ' %', 'ℹ️' FROM gkg_clean
UNION ALL
SELECT '3. Complétude (Information)', 'Persons (% Vides)', CAST(ROUND(100.0 * SUM(CASE WHEN Persons IS NULL OR Persons = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS VARCHAR) || ' %', 'ℹ️' FROM gkg_clean
UNION ALL
SELECT '3. Complétude (Information)', 'Organizations (% Vides)', CAST(ROUND(100.0 * SUM(CASE WHEN Organizations IS NULL OR Organizations = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS VARCHAR) || ' %', 'ℹ️' FROM gkg_clean

-- 4. PROFILING NUMÉRIQUE
UNION ALL
SELECT '4. Profiling Numérique', 'Tone (Min | Q1 | Médiane | Moyenne | Q3 | Max)', 
    CAST(ROUND(MIN(Tone), 1) AS VARCHAR) || ' | ' || CAST(ROUND(APPROX_QUANTILE(Tone, 0.25), 1) AS VARCHAR) || ' | ' || CAST(ROUND(APPROX_QUANTILE(Tone, 0.50), 1) AS VARCHAR) || ' | ' || CAST(ROUND(AVG(Tone), 2) AS VARCHAR) || ' | ' || CAST(ROUND(APPROX_QUANTILE(Tone, 0.75), 1) AS VARCHAR) || ' | ' || CAST(ROUND(MAX(Tone), 1) AS VARCHAR), 
    CASE WHEN MIN(Tone) >= -100 AND MAX(Tone) <= 100 THEN '✅' ELSE '⚠️' END FROM gkg_clean
UNION ALL
SELECT '4. Profiling Numérique', 'WordCount (Min | Q1 | Médiane | Moyenne | Q3 | Max)', 
    CAST(MIN(WordCount) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(WordCount, 0.25) AS INTEGER) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(WordCount, 0.50) AS INTEGER) AS VARCHAR) || ' | ' || CAST(ROUND(AVG(WordCount), 0) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(WordCount, 0.75) AS INTEGER) AS VARCHAR) || ' | ' || CAST(MAX(WordCount) AS VARCHAR), 
    CASE WHEN MIN(WordCount) >= 0 THEN '✅' ELSE '⚠️' END FROM gkg_clean


-- 5. PROFILING SÉMANTIQUE (Entités)
UNION ALL
SELECT '5. Profiling Sémantique', 'Thèmes uniques (Min | Q1 | Médiane | Moyenne | Q3 | Max)', 
    CAST(MIN(val) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(val, 0.25) AS INTEGER) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(val, 0.50) AS INTEGER) AS VARCHAR) || ' | ' || CAST(ROUND(AVG(val), 1) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(val, 0.75) AS INTEGER) AS VARCHAR) || ' | ' || CAST(MAX(val) AS VARCHAR), 
    'ℹ️' FROM (SELECT CASE WHEN EnhancedThemes = '' THEN 0 ELSE ARRAY_LENGTH(string_split(EnhancedThemes, ';')) END AS val FROM gkg_clean)
UNION ALL
SELECT '5. Profiling Sémantique', 'Personnes uniques (Min | Q1 | Médiane | Moyenne | Q3 | Max)', 
    CAST(MIN(val) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(val, 0.25) AS INTEGER) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(val, 0.50) AS INTEGER) AS VARCHAR) || ' | ' || CAST(ROUND(AVG(val), 1) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(val, 0.75) AS INTEGER) AS VARCHAR) || ' | ' || CAST(MAX(val) AS VARCHAR), 
    'ℹ️' FROM (SELECT CASE WHEN Persons = '' THEN 0 ELSE ARRAY_LENGTH(string_split(Persons, ';')) END AS val FROM gkg_clean)
UNION ALL
SELECT '5. Profiling Sémantique', 'Organisations uniques (Min | Q1 | Médiane | Moyenne | Q3 | Max)', 
    CAST(MIN(val) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(val, 0.25) AS INTEGER) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(val, 0.50) AS INTEGER) AS VARCHAR) || ' | ' || CAST(ROUND(AVG(val), 1) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(val, 0.75) AS INTEGER) AS VARCHAR) || ' | ' || CAST(MAX(val) AS VARCHAR), 
    'ℹ️' FROM (SELECT CASE WHEN Organizations = '' THEN 0 ELSE ARRAY_LENGTH(string_split(Organizations, ';')) END AS val FROM gkg_clean)

ORDER BY Categorie, Indicateur;
"""

dashboard_sante = con.execute(audit_query).df()

styled_dashboard = dashboard_sante.style.set_properties(**{'text-align': 'left'}, subset=['Categorie', 'Indicateur', 'Valeur'])\
                                        .set_properties(**{'text-align': 'center'}, subset=['Statut'])\
                                        .hide(axis="index")
display(styled_dashboard)

Categorie,Indicateur,Valeur,Statut
1. Volume & Unicité,Doublons sur GKGRECORDID,0,✅
1. Volume & Unicité,Nombre total d'articles,1136023125,ℹ️
2. Conformité,IsTranslingual (valeurs hors 0/1),0,✅
2. Conformité,URLs invalides (pas HTTP) sur Source=1,0,✅
3. Complétude (Critique),GKGRECORDID / DATE / URL (% Vides),0.0 %,✅
3. Complétude (Information),EnhancedLocations (% Vides),19.93 %,ℹ️
3. Complétude (Information),EnhancedThemes (% Vides),0.0 %,ℹ️
3. Complétude (Information),Organizations (% Vides),31.78 %,ℹ️
3. Complétude (Information),Persons (% Vides),46.14 %,ℹ️
4. Profiling Numérique,Tone (Min | Q1 | Médiane | Moyenne | Q3 | Max),-100.0 | -3.6 | -0.9 | -1.19 | 1.3 | 100.0,✅


In [8]:
print("🔍 INVESTIGATION DES GIGA-OUTLIERS (URLs COMPLÈTES)")
print("="*85)

# On demande temporairement à Pandas de ne pas tronquer le contenu des colonnes
with pd.option_context('display.max_colwidth', None):

    # 1. Les articles minuscules (Ex: WordCount = 1)
    print("\n🤏 OUTLIERS BAS : WordCount très faible (<= 10 mots)")
    display(con.execute("""
        SELECT GKGRECORDID, WordCount, DocumentIdentifier AS URL
        FROM gkg_clean 
        WHERE WordCount <= 10 
        ORDER BY WordCount ASC 
        LIMIT 5;
    """).df())

    # 2. Les articles gigantesques (Ex: WordCount > 50 000)
    print("\n🏔️ OUTLIERS HAUTS : WordCount gigantesque (> 50 000 mots)")
    display(con.execute("""
        SELECT GKGRECORDID, WordCount, DocumentIdentifier AS URL
        FROM gkg_clean 
        WHERE WordCount > 5000
        ORDER BY WordCount DESC 
        LIMIT 5;
    """).df())

    # 3. Les surcharges d'entités (Organisations / Personnes / Thèmes)
    print("\n🐙 OUTLIERS HAUTS : Surcharge d'Organisations (> 500)")
    display(con.execute("""
        SELECT 
            GKGRECORDID, 
            ARRAY_LENGTH(string_split(Organizations, ';')) AS Nb_Orgs,
            DocumentIdentifier AS URL
        FROM gkg_clean 
        WHERE Organizations != '' AND ARRAY_LENGTH(string_split(Organizations, ';')) > 100
        ORDER BY Nb_Orgs DESC 
        LIMIT 5;
    """).df())

    print("\n🧠 OUTLIERS HAUTS : Surcharge de Thèmes (> 10 000)")
    display(con.execute("""
        SELECT 
            GKGRECORDID, 
            ARRAY_LENGTH(string_split(EnhancedThemes, ';')) AS Nb_Themes,
            DocumentIdentifier AS URL
        FROM gkg_clean 
        WHERE EnhancedThemes != '' AND ARRAY_LENGTH(string_split(EnhancedThemes, ';')) > 1000
        ORDER BY Nb_Themes DESC 
        LIMIT 5;
    """).df())

🔍 INVESTIGATION DES GIGA-OUTLIERS (URLs COMPLÈTES)

🤏 OUTLIERS BAS : WordCount très faible (<= 10 mots)


,GKGRECORDID,WordCount,URL
0,20150414013000-1684,1,http://www.opednews.com/populum/linkrss.php?f=Deputy-who-shot-and-killed-in-Daily-Kos-150413-573.html
1,20150219201500-T2530,1,http://www.h-avis.no/Heder_til_Aronsen-5-62-17623.html
2,20160712000000-T2283,1,http://www.milliyet.com.tr/cesme-de-duygu-seli-magazin-2276301/
3,20150816064500-T1965,1,http://www.galvnews.com/news/article_45d9959a-43d1-11e5-a546-2ffc1660e7e4.html
4,20170726013000-T2424,1,http://www.albanyherald.com/features/arts_entertainment/albany-area-coming-up-calendar-july--aug/article_4c7c7bac-330c-5424-9fe4-c5d935781f30.html



🏔️ OUTLIERS HAUTS : WordCount gigantesque (> 50 000 mots)


,GKGRECORDID,WordCount,URL
0,20150410023000-263,747966,http://www.sloveniatimes.com/palm-sunday-celebrated-in-slovenia
1,20241028134500-T2061,707869,https://blog.wenxuecity.com/myblog/80439/202410/24877.html
2,20240906070000-336,526987,https://www.federalregister.gov/documents/2024/07/31/2024-14828/medicare-and-medicaid-programs-cy-2025-payment-policies-under-the-physician-fee-schedule-and-other
3,20150528131500-1910,469593,http://www.thetelegraphandargus.co.uk/news/broadway/11877909.display/
4,20171117181500-1820,433723,https://www.federalregister.gov/documents/2017/11/17/2017-21808/payday-vehicle-title-and-certain-high-cost-installment-loans



🐙 OUTLIERS HAUTS : Surcharge d'Organisations (> 500)


,GKGRECORDID,Nb_Orgs,URL
0,20250523193000-T105,1872,http://politics.people.com.cn/n1/2025/0523/c1001-40486648.html
1,20250218181500-1069,1670,https://www.indiainfoline.com/ipo/basis-of-allotment
2,20210226094500-T1880,1663,https://china.chinadaily.com.cn/a/202102/26/WS6038af46a3101e7ce97413d7.html
3,20210225130000-T1930,1663,http://www.farmer.com.cn/2021/02/25/99865896.html
4,20210225143000-T1955,1663,http://www.81.cn/yw/2021-02/25/content_9992033.htm



🧠 OUTLIERS HAUTS : Surcharge de Thèmes (> 10 000)


,GKGRECORDID,Nb_Themes,URL
0,20240324233000-T1369,63801,https://novini.bg/sviat/rusia/836378
1,20180115171500-1562,39466,https://www.hollywoodreporter.com/news/danny-glover-creating-a-culture-sustainable-activism-1073156
2,20160325154500-1339,34486,https://www.federalregister.gov/articles/2016/03/25/2016-04800/occupational-exposure-to-respirable-crystalline-silica
3,20201225171500-T1242,33197,https://news.day.az/azerinews_politics/1301709.html
4,20150508174500-T2460,32155,http://bbs1.people.com.cn/post/129/0/0/147536307.html


In [9]:
print("Génération pour gkg_clean...")
display(generer_statistiques(con, table_name="gkg_clean", col_source="SourceCommonName_ID", col_date="date"))

Génération pour gkg_clean...


RuntimeError: Query interrupted

In [ ]:
print("⏳ Calcul de la distribution très fine (analyse des extrêmes)...")

query_quantiles = """
WITH metrics AS (
    SELECT 
        WordCount,
        CASE WHEN Organizations = '' OR Organizations IS NULL THEN 0 ELSE ARRAY_LENGTH(string_split(Organizations, ';')) END AS OrgCount,
        CASE WHEN Persons = '' OR Persons IS NULL THEN 0 ELSE ARRAY_LENGTH(string_split(Persons, ';')) END AS PersonCount,
        CASE WHEN EnhancedThemes = '' OR EnhancedThemes IS NULL THEN 0 ELSE ARRAY_LENGTH(string_split(EnhancedThemes, ';')) END AS ThemeCount,
        CASE WHEN EnhancedLocations = '' OR EnhancedLocations IS NULL THEN 0 ELSE ARRAY_LENGTH(string_split(EnhancedLocations, ';')) END AS LocCount
    FROM gkg_clean
)
SELECT 
    -- Ajout des quantiles bas (0.0, 0.0001, 0.001, 0.005) pour voir le comportement près de zéro
    APPROX_QUANTILE(WordCount,   [0.0, 0.0001, 0.001, 0.005, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999, 0.9999, 1.0]) AS wc_q,
    APPROX_QUANTILE(OrgCount,    [0.0, 0.0001, 0.001, 0.005, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999, 0.9999, 1.0]) AS org_q,
    APPROX_QUANTILE(PersonCount, [0.0, 0.0001, 0.001, 0.005, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999, 0.9999, 1.0]) AS pers_q,
    APPROX_QUANTILE(ThemeCount,  [0.0, 0.0001, 0.001, 0.005, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999, 0.9999, 1.0]) AS theme_q,
    APPROX_QUANTILE(LocCount,    [0.0, 0.0001, 0.001, 0.005, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999, 0.9999, 1.0]) AS loc_q
FROM metrics;
"""

# Exécution de la requête
df_quantiles = con.execute(query_quantiles).df()

# Définition des labels pour nos pourcentages, adaptés à la nouvelle requête
percentiles = [
    'Min (0%)', '0.01%', '0.1%', '0.5%', '1%', '5%', '10%', '25%', 
    '50% (Med)', '75%', '90%', '95%', '99%', '99.5%', '99.9%', '99.99%', 'Max'
]

# Création d'un DataFrame propre en transposant les résultats
df_fine_distribution = pd.DataFrame({
    'Mots (WordCount)': df_quantiles['wc_q'][0],
    'Localisations uniques': df_quantiles['loc_q'][0],
    'Organisations uniques': df_quantiles['org_q'][0],
    'Personnes uniques': df_quantiles['pers_q'][0],
    'Thèmes uniques': df_quantiles['theme_q'][0]
}, index=percentiles).T

# Affichage avec un dégradé de couleurs (heatmap) par ligne
display(
    df_fine_distribution.style.format("{:,.0f}")
    .background_gradient(cmap='YlOrRd', axis=1)
    .set_caption("📊 Distribution granulaire des métadonnées (repérage des Outliers Hauts et Bas)")
)

In [ ]:
print("⏳ Calcul du quantile exact pour le seuil de 150 mots...")

query_exact_quantile = """
    SELECT 
        COUNT(*) AS total_articles,
        SUM(CASE WHEN WordCount <= 150 THEN 1 ELSE 0 END) AS articles_exclus_bas,
        ROUND(100.0 * SUM(CASE WHEN WordCount <= 150 THEN 1 ELSE 0 END) / COUNT(*), 2) AS quantile_150_mots
    FROM gkg_clean;
"""

df_quantile = con.execute(query_exact_quantile).df()

# Formatage de l'affichage
styler_quantile = (
    df_quantile.style
    .format({
        "total_articles": "{:,.0f}",
        "articles_exclus_bas": "{:,.0f}",
        "quantile_150_mots": "{:.2f} %"
    })
    .set_properties(**{"text-align": "center", "font-weight": "bold", "font-size": "14px"})
)

display(styler_quantile)

In [ ]:
print("🔍 INVESTIGATION DES ARTICLES LONGS (WordCount 2000 - 2500) - Tirage récent et aléatoire")
print("="*85)

query_long_articles = """
    SELECT 
        GKGRECORDID, 
        strptime(substr(CAST(DATE AS VARCHAR), 1, 8), '%Y%m%d')::DATE AS Date_Article,
        WordCount,
        CASE WHEN Organizations = '' OR Organizations IS NULL THEN 0 ELSE ARRAY_LENGTH(string_split(Organizations, ';')) END AS Nb_Orgs,
        CASE WHEN Persons = '' OR Persons IS NULL THEN 0 ELSE ARRAY_LENGTH(string_split(Persons, ';')) END AS Nb_Persons,
        DocumentIdentifier AS URL
    FROM gkg_clean 
    WHERE WordCount BETWEEN 100 AND 150
      -- Filtre sur les articles récents (ex: depuis le 1er janvier 2025)
      AND CAST(DATE AS VARCHAR) >= '20250101000000'
    -- Mélange aléatoire des résultats pour avoir de la diversité
    ORDER BY RANDOM()
    LIMIT 50;
"""

# Forcer Pandas à afficher l'URL en entier
with pd.option_context('display.max_colwidth', None):
    df_longs = con.execute(query_long_articles).df()
    display(df_longs.style.set_properties(**{'text-align': 'left'}))

In [ ]:
# Votre liste de sites de référence
SITES_RECONNUS = [
    "wsj.com", "nytimes.com", "washingtonpost.com", "chicagotribune.com",
    "telegraph.co.uk", "theglobeandmail.com", "ft.com", "theguardian.com",
    "latimes.com", "usatoday.com", "bloomberg.com", "reuters.com",
    "economist.com", "forbes.com", "nikkei.com", "handelsblatt.com",
    "ilsole24ore.com", "cincodias.elpais.com", "expansion.com",
    "financialpost.com", "lesechos.fr", "latribune.fr", "challenges.fr",
    "lemonde.fr", "capital.fr", "lecho.be", "agefi.com", "apnews.com",
    "afp.com", "bbc.co.uk", "bbc.com", "cnn.com", "time.com",
    "thetimes.co.uk", "smh.com.au", "torontostar.com", "scmp.com",
    "thehindu.com", "elpais.com", "elmundo.es", "spiegel.de",
    "zeit.de", "sueddeutsche.de", "faz.net", "corriere.it",
    "repubblica.it", "lastampa.it", "clarin.com", "lanacion.com.ar",
    "oglobo.globo.com", "eluniversal.com.mx", "lefigaro.fr",
    "liberation.fr", "leparisien.fr", "la-croix.com", "lexpress.fr",
    "lepoint.fr", "nouvelobs.com", "20minutes.fr",
    "courrierinternational.com", "franceinfo.fr", "ouest-france.fr",
    "sudouest.fr", "lavoixdunord.fr", "lorientlejour.com",
    "elwatan.com", "le360.ma"
]

# Préparation des chaînes pour la clause IN de SQL (gère avec et sans 'www.')
sites_sql = ", ".join([f"'{s}'" for s in SITES_RECONNUS])
sites_sql_www = ", ".join([f"'www.{s}'" for s in SITES_RECONNUS])

print("⏳ Calcul des statistiques descriptives (WordCount) sur le Gold Standard...")

query_gold_stats = f"""
    SELECT 
        m.SourceCommonName AS Media,
        COUNT(g.GKGRECORDID) AS Total_Articles,
        MIN(g.WordCount) AS Min_Mots,
        CAST(APPROX_QUANTILE(g.WordCount, 0.25) AS INTEGER) AS Q1_Mots,
        CAST(MEDIAN(g.WordCount) AS INTEGER) AS Mediane_Mots,
        ROUND(AVG(g.WordCount), 0) AS Moyenne_Mots,
        CAST(APPROX_QUANTILE(g.WordCount, 0.75) AS INTEGER) AS Q3_Mots,
        ROUND(STDDEV(g.WordCount), 0) AS Ecart_Type,
        MAX(g.WordCount) AS Max_Mots
    FROM gkg_clean g
    JOIN src_map m ON g.SourceCommonName_ID = m.SourceCommonName_ID
    WHERE m.SourceCommonName IN ({sites_sql}) 
       OR m.SourceCommonName IN ({sites_sql_www})
    GROUP BY m.SourceCommonName
    ORDER BY Moyenne_Mots DESC;
"""

# Exécution de la requête
df_gold_stats = con.execute(query_gold_stats).df()

# Formatage visuel pour repérer facilement les extrêmes
styler_gold = (
    df_gold_stats.style
    .format({
        "Total_Articles": "{:,.0f}",
        "Min_Mots": "{:,.0f}",
        "Q1_Mots": "{:,.0f}",
        "Mediane_Mots": "{:,.0f}",
        "Moyenne_Mots": "{:,.0f}",
        "Q3_Mots": "{:,.0f}",
        "Ecart_Type": "{:,.0f}",
        "Max_Mots": "{:,.0f}"
    })
    .background_gradient(subset=["Min_Mots"], cmap="Reds", vmin=0, vmax=100) # Met en rouge les Mins suspects
    .background_gradient(subset=["Mediane_Mots", "Moyenne_Mots"], cmap="Blues")
    .background_gradient(subset=["Max_Mots"], cmap="Oranges") # Met en orange les Maxs très élevés
    .set_properties(subset=["Media"], **{"font-weight": "bold", "text-align": "left"})
)

display(styler_gold)

In [ ]:
print("⏳ Création de la vue 'gkg_clean_2' (Filtrage simple sur le WordCount)...")

con.execute("""
    CREATE OR REPLACE VIEW gkg_clean_2 AS
    SELECT *
    FROM gkg_clean
    WHERE WordCount BETWEEN 150 AND 5500;
""")

print("✔ Vue 'gkg_clean_2' prête ! L'échantillon est désormais restreint aux articles contenant entre 150 et 7 500 mots.")

In [ ]:
print("Génération pour gkg_clean_2...")
display(generer_statistiques(con, table_name="gkg_clean_2", col_source="SourceCommonName_ID", col_date="date"))

In [ ]:
print("🔍 ANALYSE DES SOURCES INTÉGRALEMENT SUPPRIMÉES PAR LE FILTRE")
print("="*85)

query_sources_perdues = """
    -- 1. On isole les IDs des médias qui ont complètement disparu
    WITH sources_perdues AS (
        SELECT DISTINCT SourceCommonName_ID FROM gkg_clean
        EXCEPT
        SELECT DISTINCT SourceCommonName_ID FROM gkg_clean_2
    )
    
    -- 2. On calcule les stats sur gkg_clean (avant le filtre) pour ces médias
    SELECT 
        COALESCE(m.SourceCommonName, 'ID Inconnu (' || g.SourceCommonName_ID || ')') AS Media,
        COUNT(g.GKGRECORDID) AS Total_Articles,
        MIN(g.WordCount) AS Min_Mots,
        CAST(APPROX_QUANTILE(g.WordCount, 0.25) AS INTEGER) AS Q1_Mots,
        CAST(MEDIAN(g.WordCount) AS INTEGER) AS Mediane_Mots,
        ROUND(AVG(g.WordCount), 0) AS Moyenne_Mots,
        CAST(APPROX_QUANTILE(g.WordCount, 0.75) AS INTEGER) AS Q3_Mots,
        MAX(g.WordCount) AS Max_Mots
    FROM gkg_clean g
    INNER JOIN sources_perdues s ON g.SourceCommonName_ID = s.SourceCommonName_ID
    LEFT JOIN src_map m ON g.SourceCommonName_ID = m.SourceCommonName_ID
    GROUP BY m.SourceCommonName, g.SourceCommonName_ID
    ORDER BY Total_Articles DESC;
"""

# Exécution
df_sources_perdues = con.execute(query_sources_perdues).df()

print(f"⚠️ Nombre total de médias 100% exclus : {len(df_sources_perdues)}")

if len(df_sources_perdues) > 0:
    # Formatage visuel pour voir d'un coup d'œil de quel côté le filtre a coupé (trop court ou trop long)
    styler_perdus = (
        df_sources_perdues.style
        .format({
            "Total_Articles": "{:,.0f}",
            "Min_Mots": "{:,.0f}",
            "Q1_Mots": "{:,.0f}",
            "Mediane_Mots": "{:,.0f}",
            "Moyenne_Mots": "{:,.0f}",
            "Q3_Mots": "{:,.0f}",
            "Max_Mots": "{:,.0f}"
        })
        # Mise en évidence en rouge si le Max est sous 150 (coupés par le bas) 
        # ou si le Min est au-dessus de 7500 (coupés par le haut)
        .apply(lambda x: ['color: red; font-weight: bold' if v < 150 else '' for v in x], subset=['Max_Mots'])
        .apply(lambda x: ['color: red; font-weight: bold' if v > 7500 else '' for v in x], subset=['Min_Mots'])
        .set_properties(subset=["Media"], **{"font-weight": "bold", "text-align": "left"})
    )
    
    with pd.option_context("display.max_rows", 200):
        display(styler_perdus)
else:
    print("Bonne nouvelle : aucun média n'a été intégralement supprimé par ce filtre !")